In [1]:
from pathlib import Path
from tqdm import tqdm
import torch
from lerobot.datasets.utils import cycle

from lerobot.configs.types import FeatureType
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.datasets.utils import dataset_to_policy_features
from lerobot.policies.smolandfast.configuration_smolandfast import SMOLANDFASTConfig
from lerobot.policies.smolandfast.modeling_smolandfast import SMOLANDFASTPolicy

/Users/olegbalakhnov/Documents/RLC/.lerobot/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
output_directory = Path("outputs/train/example_pusht")
output_directory.mkdir(parents=True, exist_ok=True)

device = torch.device("cpu")

In [3]:
DATASET_PATH = "lerobot/pusht"

dataset_metadata = LeRobotDatasetMetadata(DATASET_PATH)
features = dataset_to_policy_features(dataset_metadata.features)
output_features = {key: ft for key, ft in features.items() if ft.type is FeatureType.ACTION}
input_features = {key: ft for key, ft in features.items() if key not in output_features}

cfg = SMOLANDFASTConfig(input_features=input_features,
                        output_features=output_features)

delta_timestamps = {
        "action": [i / dataset_metadata.fps for i in cfg.action_delta_indices],
    }

# We can then instantiate the dataset with these delta_timestamps configuration.
dataset = LeRobotDataset(DATASET_PATH, delta_timestamps=delta_timestamps)

dataloader = torch.utils.data.DataLoader(
    dataset,
    num_workers=0,
    batch_size=1,
    shuffle=True,
    pin_memory=device.type != "cpu",
    drop_last=True,
)
dl_iter = cycle(dataloader)


In [4]:
policy = SMOLANDFASTPolicy(cfg,
                           dataset_stats=dataset_metadata.stats)
policy.train()
policy.to(device)

optimizer = torch.optim.Adam(policy.parameters(), lr=5e-5)

/Users/olegbalakhnov/Documents/RLC/.lerobot/lib/python3.11/site-packages/transformers/models/auto/modeling_auto.py:2242: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 11.79it/s]


Idefics3Processor:
- image_processor: Idefics3ImageProcessor {
  "do_convert_rgb": true,
  "do_image_splitting": true,
  "do_normalize": true,
  "do_pad": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "Idefics3ImageProcessor",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "max_image_size": {
    "longest_edge": 512
  },
  "processor_class": "Idefics3Processor",
  "resample": 1,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "longest_edge": 2048
  }
}

- tokenizer: GPT2TokenizerFast(name_or_path='HuggingFaceTB/SmolVLM-256M-Instruct', vocab_size=49152, model_max_length=8192, is_fast=True, padding_side='right', truncation_side='left', special_tokens={'bos_token': '<|im_start|>', 'eos_token': '<end_of_utterance>', 'unk_token': '<|endoftext|>', 'pad_token': '<|im_end|>', 'additional_special_tokens': ['<fake_token_around_image>', '<image>', '<end_of_utterance>']}, clean_up_tokenization_spaces=False, a

In [5]:
batch = next(dl_iter)

for step in tqdm(range(50)):

    batch = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
    loss, _ = policy.forward(batch)
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    print(f"step: {step} loss: {loss.item():.3f}")

objc[30994]: Class AVFFrameReceiver is implemented in both /Users/olegbalakhnov/Documents/RLC/.lerobot/lib/python3.11/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x135c443a8) and /opt/homebrew/Cellar/ffmpeg/6.1.1_4/lib/libavdevice.60.3.100.dylib (0x14e5d4370). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[30994]: Class AVFAudioReceiver is implemented in both /Users/olegbalakhnov/Documents/RLC/.lerobot/lib/python3.11/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x135c443f8) and /opt/homebrew/Cellar/ffmpeg/6.1.1_4/lib/libavdevice.60.3.100.dylib (0x14e5d43c0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
  0%|          | 0/50 [00:00<?, ?it/s]


NotImplementedError: Module [SMOLANDFASTVision] is missing the required "forward" function

In [ ]:
batch_norm = policy.normalize_inputs(batch)
batch_norm = policy.normalize_targets(batch_norm)

decoded_actions = policy.model.generate_actions(batch_norm)
error:torch.tensor = torch.sqrt((decoded_actions - batch_norm["action"])**2)

print(f"RMSE {(error.mean(dim=1)*100).tolist()}%")

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


RMSE [[1.5230470895767212, 2.0934603214263916], [1.6779065132141113, 1.287825107574463]]%


/Users/olegbalakhnov/Documents/RLC/lerobot-experiments/src/lerobot/policies/smolandfast/modeling_smolandfast.py:416: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:257.)
  decoded_actions = torch.tensor([
